
Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [4]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model("groq:qwen/qwen3-32b")

response = model.invoke("Why do parrots talk?")

response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re birds, right? Some species like the African Grey, Macaws, and Amazon Parrots are known for their ability to mimic human speech. But why do they do that?\n\nFirst, maybe it\'s about communication. Parrots are social animals, so talking could be a way they interact with each other or with humans. I remember reading that some animals use vocalizations to communicate within their group, maybe for things like finding food or warning about danger. Parrots might be using speech as a way to bond with their human companions. But is that the main reason?\n\nAnother angle: mimicry. Parrots are known for imitating sounds. They might not understand the words they\'re saying but can copy the sounds. So, talking could just be a learned behavior. But why would they learn to talk specifically? Maybe because they\'re in an environment where humans are talking a lot, s

In [5]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the current weather in a given location."""
    return f"The current weather in {location} is sunny with a high of 25°C."

model_with_tools=model.bind_tools([get_weather])

In [6]:

response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which in this case is Boston. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments. No other parameters are needed. Just the location. Alright, that should do it.\n", 'tool_calls': [{'id': 'm3sqq5vwm', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 156, 'total_tokens': 261, 'completion_time': 0.166853941, 'completion_tokens_details': {'reasoning_tokens': 81}, 'prompt_time': 0.006241699, 'prompt_tokens_details': None, 'queue_time': 0.04911163, 'total_time': 0.17309564}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'too

Tool Execution Loops

In [7]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The current weather in Boston is sunny with a high of 25°C. A perfect day to enjoy outdoor activities! 🌞
